# SignalMint demo

**One tiny autoregressive model, two products** on 1-D signals:

1. **Label-free anomaly detection** — improbable samples (high NLL) are anomalies.
2. **Neural compression** — the same per-sample probabilities drive an arithmetic coder.

This notebook runs end-to-end on **synthetic bearing vibration** (no download).
Swap `get_synthetic_records(...)` for `load_records(source='cwru')` for real data.

In [ ]:
import numpy as np
from signalmint.config import DataConfig, ModelConfig, TrainConfig
from signalmint.data.cwru import get_synthetic_records
from signalmint.data.dataset import records_to_frames
from signalmint.train import train_model

records = get_synthetic_records(n_normal=8, n_fault=8, n_samples=8000, seed=0)
data_cfg = DataConfig(frame_length=256, num_bins=16, hop_length=128)
frames = records_to_frames(records, data_cfg)
normal = frames.filter_label(0)
print('frames:', len(frames), 'normal:', len(normal), 'fault:', len(frames) - len(normal))

## Train on healthy data only
The model never sees a fault during training, yet learns to flag them.

In [ ]:
model_cfg = ModelConfig(num_bins=16, residual_channels=8, skip_channels=8, num_layers=4, dilation_cycle=4)
train_cfg = TrainConfig(batch_size=16, epochs=5, learning_rate=3e-3, val_fraction=0.2, seed=0)
result = train_model(normal, model_cfg, train_cfg)
model = result.model
print(f'best val NLL: {result.best_val_nll:.3f} nats  ->  {result.bits_per_sample:.3f} bits/sample')
print(f'params: {model.num_parameters():,}  receptive field: {model.receptive_field}')

## Product 1: anomaly detection

In [ ]:
from signalmint.anomaly.detector import evaluate_anomaly

report, scores = evaluate_anomaly(model, frames, target_fa=0.05)
print(f'ROC AUC:              {report.auc:.3f}')
print(f'detection @ 5% FA:    {report.detection_rate:.3f}')

## Product 2: compression (same model)

In [ ]:
from signalmint.compress.coder import encode_frames, decode_frame
from signalmint.compress.baselines import gzip_bits_per_sample, order0_entropy_bits_per_sample, raw_bits_per_sample

subset = normal.symbols[:16]
comp = encode_frames(model, subset, total_bits=16)
decoded = decode_frame(model, encode_frames(model, subset[:1], 16).data, length=subset.shape[1], total_bits=16)
order0_bits = order0_entropy_bits_per_sample(subset, 16)
print(f'neural:   {comp.bits_per_sample:.3f} bits/sample')
print(f'order-0:  {order0_bits:.3f} bits/sample')
print(f'uniform:  {raw_bits_per_sample(16):.3f} bits/sample')
print('round-trip exact:', bool(np.array_equal(decoded, subset[0])))

## INT8 quantization + edge footprint
The integer reference matches the float model, and reports a tiny footprint.

In [ ]:
import torch
from signalmint.quantize.quantizer import quantize_model
from signalmint.quantize.int_infer import IntegerRuntime
from signalmint.quantize.footprint import analyze_footprint

qm = quantize_model(model)
rt = IntegerRuntime(qm)
frame = normal.symbols[0]
with torch.no_grad():
    fl = model.forward(torch.from_numpy(frame[None, :].astype(np.int64)))[0].numpy()
il = rt.logits(frame)
print('float/int argmax agreement:', float(np.mean(fl.argmax(1) == il.argmax(1))))

fp = analyze_footprint(qm)
print(f'ROM: {fp.rom_kb:.1f} KB   streaming RAM: {fp.streaming_ram_kb:.2f} KB   MACs/sample: {fp.macs_per_sample:,}')

To export the model to the C runtime and verify bit-exact parity, run:

```bash
python scripts/export_c.py --checkpoint artifacts/model.pt --out runtime/model_data.h
python scripts/parity.py   --checkpoint artifacts/model.pt
```